In [ ]:
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 60)
np.random.seed(0)
print("pandas", pd.__version__)

In [ ]:
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
display(df.head(3))
print("shape:", df.shape)
display(df.describe())
df.info()
df.isnull().sum()

In [ ]:
print("distinct country:", df["Country"].nunique())
print(df["Country"].value_counts())

print("distinct stockcode:", df["StockCode"].nunique())
print(df["StockCode"].value_counts())

print("distinct description:", df["Description"].nunique())
print(df["Description"].value_counts())

print(df["StockCode"].value_counts(dropna=False)) # StockCode and Description are the Product code/name
print(df["Description"].value_counts(dropna=False))

In [ ]:
display(df.describe())

print("Num of negative Quantity: ", df[df["Quantity"] < 0].shape[0])
print("Num of negative or zero UnitPrice: ", df[df["UnitPrice"] <= 0].shape[0])


In [ ]:
region_lookup_table = {
    "United Kingdom": "UK&IE",
    "Germany": "Western Europe",
    "France": "Western Europe",
    "EIRE": "UK&IE",
    "Spain": "Southern Europe",
    "Netherlands": "Western Europe",
    "Belgium": "Western Europe",
    "Switzerland": "Western Europe",
    "Portugal": "Southern Europe",
    "Australia": "Oceania",
    "Norway": "Northern Europe",
    "Italy": "Southern Europe",
    "Channel Islands": "UK&IE",
    "Finland": "Northern Europe",
    "Cyprus": "Southern Europe",
    "Sweden": "Northern Europe",
    "Unspecified": "Unknown",
    "Austria": "Western Europe",
    "Denmark": "Northern Europe",
    "Japan": "East Asia",
    "Poland": "Eastern Europe",
    "Israel": "Middle East",
    "USA": "North America",
    "Hong Kong": "East Asia",
    "Singapore": "Southeast Asia",
    "Iceland": "Northern Europe",
    "Canada": "North America",
    "Greece": "Southern Europe",
    "Malta": "Southern Europe",
    "United Arab Emirates": "Middle East",
    "European Community": "Europe",
    "RSA": "Africa",
    "Lebanon": "Middle East",
    "Lithuania": "Eastern Europe",
    "Brazil": "South America",
    "Czech Republic": "Eastern Europe",
    "Bahrain": "Middle East",
    "Saudi Arabia": "Middle East"
}

In [ ]:
# display(df.iloc[0:3])
# df["LineItem"] = df.index
# display(df.loc[:, ["Quantity", "UnitPrice"]].head())
# df2 = df.set_index(df.index)
# display(df2.loc[0:3])

display(df.iloc[0:3])
df.index = pd.Index([f"L{i:06d}" for i in range(len(df))], name="line-item")
display(df.loc["L000005"])

In [ ]:
print("Cancellations:", df[df["InvoiceNo"].str.startswith("C")].shape[0])
print("Num of negative or zero Quantity: ", df[df["Quantity"] <= 0].shape[0])
print("Num of negative or zero UnitPrice: ", df[df["UnitPrice"] <= 0].shape[0]) # I think I did it previuosly, but I'll just follow instruction


In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["Revenue"] = df["Revenue"].round(3)
sorted_in_largest_bulk_orders = df.sort_values("Quantity", ascending=False)
print("Largest Bulk Orders: \n", sorted_in_largest_bulk_orders.head(1))
sorted_in_largest_returns = df.sort_values("Revenue", ascending=False)
print("Largest returns: \n", sorted_in_largest_returns.head(1))

In [ ]:
# Begin of Phoenix
data = df

In [ ]:
# Phase C
# 3.6
data["Country"] = data["Country"].replace({ "EIRE": "Ireland", "RSA": "South Africa", "Unspecified": pd.NA })  # type: ignore
print("EIRE -> Ireland, RSA -> South Africa, Unspecified -> missing")
print(data["Country"].value_counts(dropna=False).head())

In [ ]:
# 3.7
COLUMN_MAP={
    "InvoiceNo": "invoice_no", "StockCode": "stock_code",
    "Description": "description", "Quantity": "quantity",
    "InvoiceDate": "invoice_date", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
}
data.rename(columns=COLUMN_MAP, inplace=True)
print(data.columns)


In [ ]:
# 3.10
# Custom ID
print("Number of missing customer IDs:", data["customer_id"].isnull().sum())
data["customer_id"] = data["customer_id"].fillna(-1)
# Empty descriptions
# mask = data["description"].str.strip().eq("")
print("Number of empty descriptions:", data["description"].isnull().sum())
data["description"] = data["description"].fillna(pd.NA)


In [ ]:
# 3.11
# [TODO]?
# data.dropna(subset=["description"])

In [ ]:
# 3.12
# is_cancelled bad_price bad_qty
is_cancelled = data["invoice_no"].astype(str).str.startswith("C")
bad_qty = data["quantity"] <= 0
bad_price = data["unit_price"] <= 0

print(f"Data size before dropping invalid rows: {len(data)}")
data = data.drop(index=data[is_cancelled | bad_price | bad_qty].index)
not_product = (~data["stock_code"].astype(str).str.match(r"^\d{5}"))
data = data.drop(index=data[not_product].index)
data = data.dropna(subset=["description"])
print(f"Data size after dropping invalid rows: {len(data)}")

In [ ]:
# 3.13
duplicate_number = int(data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)
print("Exact duplicate rows removed:", duplicate_number)
print("Rows remaining:", len(data))

In [ ]:
# Phase D
# 3.18
data.rename(columns={"Revenue": "revenue"}, inplace=True)
# data.columns

data["description"] = data["description"].apply(lambda s: str(s).strip().title())
data["is_cancelled"] = data["invoice_no"].astype(str).str.startswith("C")

data.head()

In [ ]:
# 3.17
import time

s = time.time()
cleaned_desc_list = []
unique_data = data["description"].unique()
for des in unique_data:
    cleaned = str(des).strip().title()
    cleaned_desc_list.append(cleaned)
e = time.time()
print(f"for-loop: time cost in {e - s:.4f} seconds")

s = time.time()
cleaned_desc_list = [
    str(des).strip().title() for des in data["description"].unique()
]
e = time.time()
print(f"list comprehension: time cost in {e - s:.4f} seconds")

s = time.time()
data["description"] = data["description"].apply(lambda s: str(s).strip().title())
e = time.time()
print(f"apply: time cost in {e - s:.4f} seconds")

# In my timing test, approach `list comprehension` > `for-loop` > `apply`, in terms of speed.
# This might be reasonable because apply often still executes Python-level operations element by element.

In [ ]:
# 3.14
rev_by_country = data.groupby("country")["revenue"].sum().sort_values(ascending=False)
print("Revenue by country:", rev_by_country.head(), sep="\n")

print(f"\n{'-' * 25}\n")

orders_per_customer = data[data["customer_id"] != -1].groupby("customer_id")["invoice_no"].nunique().sort_values(ascending=False)
print("Orders per customer:", orders_per_customer.head(), sep="\n")

In [ ]:
# 3.16
by_country = data.groupby("country").agg(
    total_revenue=("revenue", "sum"),
    mean_line_value=("revenue", "mean"),
    transactions=("invoice_no", "nunique"),
).sort_values("total_revenue", ascending=False)
# by_country.head()

mask = data["customer_id"] != -1
by_customer = data[mask].groupby("customer_id").agg(
    total_revenue=("revenue", "sum"),
    mean_line_value=("revenue", "mean"),
    transactions=("invoice_no", "nunique"),
).sort_values("transactions", ascending=False)
by_customer.head()

In [ ]:
# 3.19
data["invoice_date"] = pd.to_datetime(data["invoice_date"])  # Also for 3.15

valid_customers = data[data["customer_id"] != -1]

def customer_summary(cs):
    return pd.Series({
        "total_spend":   cs["revenue"].sum(),
        "n_orders":      cs["invoice_no"].nunique(),
        "active_months": cs["invoice_date"].dt.to_period("M").nunique(),
    })

per_customer = valid_customers.groupby("customer_id").apply(customer_summary)
per_customer["n_orders"] = per_customer["n_orders"].astype(int)
per_customer["active_months"] = per_customer["active_months"].astype(int)
per_customer.sort_values("n_orders", ascending=False).head()


In [ ]:
# 3.15
ts = data.set_index("invoice_date").sort_index()

# Monthly
monthly = ts["revenue"].resample("MS").sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly.index, monthly.values, marker="o")
ax.set_title("Monthly revenue"); ax.set_xlabel("Month"); ax.set_ylabel("Revenue")
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(r"monthly_revenue.png")

# Weekly
weekly = ts["revenue"].resample("W").sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(weekly.index, weekly.values, marker="o")
ax.set_title("Weekly revenue"); ax.set_xlabel("Week"); ax.set_ylabel("Revenue")
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(r"weekly_revenue.png")
plt.show()

In [ ]:
# END of Phoenix
df = data

In [ ]:
first_half = df[df["invoice_date"].dt.month <= 6]
second_half = df[df["invoice_date"].dt.month > 6]
pd.concat([first_half, second_half])

In [ ]:
region_lookup_df = pd.DataFrame(list(region_lookup_table.items()), columns=["country", "region"])

df_left = df.merge(region_lookup_df, on="country", how="left")
df_inner = df.merge(region_lookup_df, on="country", how="inner")

display(df_left.head())
display(df_inner.head())

In [ ]:
# Save the cleaned data to a new CSV file
data.to_csv(r"clean_online_retail.csv", index=False)